<a href="https://colab.research.google.com/github/RadhikaDeshpande1010/pyspark-rdd-sales-analytics/blob/main/pyspark_rdd_sales_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark RDD — Sales Data Analysis (Q1–Q35)

This notebook demonstrates core PySpark RDD operations — `map()`, `filter()`, `flatMap()`, and `reduceByKey()` — applied to a retail sales dataset across **35 analytical exercises**.

**Dataset:** `sales_data.txt` &nbsp;|&nbsp; **Schema:** `order_id, customer_id, product, category, city, quantity, unit_price`

**Author:** Radhika Deshpande &nbsp;|&nbsp; **GitHub:** [RadhikaDeshpande1010](https://github.com/RadhikaDeshpande1010)

---
## Setup — Initialize Spark Session and Context

In [ ]:
from pyspark.sql import SparkSession

spark_session = SparkSession.builder \
    .master("local") \
    .appName("PySpark RDD Sales Analysis") \
    .getOrCreate()

print(spark_session)

In [ ]:
spark_context = spark_session.sparkContext
print(spark_context)

<SparkContext master=local appName=PySpark RDD Sales Analysis>


---
## Data Ingestion

In [ ]:
# Load raw sales data from text file into an RDD
raw_sales_rdd = spark_context.textFile("/content/sample_data/sales_data.txt")

---
## Q1 — Parse Records into Structured Lists
Load the `sales_data.txt` file into an RDD and convert each record into a list by splitting on commas.

In [ ]:
# Split each raw line on commas to produce a list of fields per record
parsed_sales_rdd = raw_sales_rdd.map(lambda raw_line: [field.strip() for field in raw_line.strip().split(",")])

parsed_sales_rdd.take(5)

[['1', 'C112', 'Printer', 'Electronics', 'Chennai', '5', '12000'],
 ['2', 'C137', 'Headphones', 'Electronics', 'Delhi', '5', '2000'],
 ['3', 'C150', 'Phone', 'Electronics', 'Patna', '1', '30000'],
 ['4', 'C113', 'Tablet', 'Electronics', 'Hyderabad', '5', '25000'],
 ['5', 'C150', 'Bed', 'Furniture', 'Hyderabad', '4', '18000']]

---
## Q2 — Total Price per Order
Using `map()`, calculate the total price for each record (`quantity × unit_price`).  
Output: `(order_id, total_price)`.

In [ ]:
# Compute total revenue per order: quantity (index 5) * unit_price (index 6)
order_total_price_rdd = parsed_sales_rdd.map(
    lambda record: (record[0], int(record[5]) * int(record[6]))
)

print(order_total_price_rdd.collect())

[('1', 60000), ('2', 10000), ('3', 30000), ('4', 125000), ('5', 72000), ('6', 36000), ('7', 14000), ('8', 110000), ('9', 8000), ('10', 150000), ('11', 2000), ('12', 25000), ('13', 120000), ('14', 21000), ('15', 24000), ('16', 36000), ('17', 54000), ('18', 8000), ('19', 100000), ('20', 72000), ('21', 25000), ('22', 8000), ('23', 48000), ('24', 16000), ('25', 125000), ('26', 100000), ('27', 36000), ('28', 15000), ('29', 125000), ('30', 90000), ('31', 60000), ('32', 14000), ('33', 45000), ('34', 10000), ('35', 60000), ('36', 6000), ('37', 30000), ('38', 12000), ('39', 54000), ('40', 10000), ('41', 15000), ('42', 12000), ('43', 7000), ('44', 80000), ('45', 100000), ('46', 21000), ('47', 275000), ('48', 100000), ('49', 15000), ('50', 20000), ('51', 30000), ('52', 60000), ('53', 28000), ('54', 50000), ('55', 220000), ('56', 48000), ('57', 18000), ('58', 165000), ('59', 72000), ('60', 2000), ('61', 25000), ('62', 90000), ('63', 120000), ('64', 60000), ('65', 60000), ('66', 125000), ('67', 250

---
## Q3 — Electronics Orders
Using `filter()`, extract the `order_id` for all orders where the category is **Electronics**.

In [ ]:
# Filter records where category is Electronics
electronics_category_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], record[3])) \
    .filter(lambda order_category: order_category[1] == "Electronics") \
    .map(lambda order_category: order_category[0])

print(electronics_category_orders_rdd.collect())

['1', '2', '3', '4', '8', '10', '11', '12', '13', '15', '16', '18', '19', '21', '22', '23', '25', '26', '27', '28', '29', '30', '31', '33', '34', '35', '36', '37', '38', '40', '41', '42', '45', '47', '48', '49', '51', '54', '55', '56', '58', '60', '61', '63', '65', '66', '67', '69', '70', '71', '72', '73', '74', '77', '78', '79', '80', '81', '82', '87', '88', '89', '90', '95', '96', '97']


---
## Q4 — Orders with Quantity Greater Than 2
Using `filter()`, find all records where the quantity sold is greater than 2.

In [ ]:
# Filter records where quantity (index 5) > 2
high_quantity_orders_rdd = parsed_sales_rdd.filter(
    lambda record: int(record[5]) > 2
)

high_quantity_orders_rdd.count()

60

---
## Q5 — Total Sales Amount per Product
Using `map()` and `reduceByKey()`, calculate the total sales amount for each product.

In [ ]:
# Key by product name (index 2), value = quantity * unit_price
total_sales_per_product_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda sales_a, sales_b: sales_a + sales_b)

print(total_sales_per_product_rdd.collect())

[('Printer', 408000), ('Headphones', 58000), ('Phone', 1260000), ('Tablet', 1350000), ('Bed', 666000), ('Table', 161000), ('Laptop', 880000), ('Chair', 52000), ('Monitor', 390000), ('Sofa', 380000)]


---
## Q6 — Total Quantity Sold per Product
Using `map()` and `reduceByKey()`, calculate the total quantity sold for each product.

In [ ]:
# Key by product name (index 2), value = quantity (index 5)
total_quantity_per_product_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], int(record[5]))) \
    .reduceByKey(lambda qty_a, qty_b: qty_a + qty_b)

print(total_quantity_per_product_rdd.collect())

[('Printer', 34), ('Headphones', 29), ('Phone', 42), ('Tablet', 54), ('Bed', 37), ('Table', 23), ('Laptop', 16), ('Chair', 13), ('Monitor', 26), ('Sofa', 19)]


---
## Q7 — Orders with Unit Price Greater Than 20,000
Using `filter()`, find all orders where the unit price exceeds 20,000.

In [ ]:
# Map to (order_id, unit_price) then filter on unit_price > 20000
high_unit_price_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], int(record[6]))) \
    .filter(lambda order_price: order_price[1] > 20000)

print(high_unit_price_orders_rdd.collect())

[('3', 30000), ('4', 25000), ('8', 55000), ('10', 30000), ('12', 25000), ('13', 30000), ('19', 25000), ('21', 25000), ('25', 25000), ('26', 25000), ('29', 25000), ('30', 30000), ('37', 30000), ('45', 25000), ('47', 55000), ('48', 25000), ('54', 25000), ('55', 55000), ('58', 55000), ('61', 25000), ('63', 30000), ('66', 25000), ('67', 25000), ('69', 30000), ('70', 25000), ('77', 30000), ('78', 30000), ('80', 55000), ('82', 30000), ('87', 30000), ('88', 25000), ('89', 25000), ('96', 25000), ('97', 30000)]


---
## Q8 — Total Revenue per City
Using `map()` and `reduceByKey()`, calculate the total sales **revenue** (`quantity × unit_price`) for each city.


In [ ]:
# Key by city (index 4), value = quantity * unit_price
total_revenue_per_city_rdd = parsed_sales_rdd \
    .map(lambda record: (record[4], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_city_rdd.collect())

[('Chennai', 687000), ('Delhi', 712000), ('Patna', 294000), ('Hyderabad', 597000), ('Mumbai', 903000), ('Bangalore', 1001000), ('Pune', 1157000), ('Kolkata', 254000)]


---
## Q9 — Furniture Orders with Quantity Greater Than 1
Using `filter()`, extract all orders where the category is **Furniture** and the quantity is greater than 1.

In [ ]:
# Map to (order_id, category, quantity), filter on Furniture + qty > 1, then project to (order_id, quantity)
furniture_bulk_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], record[3], int(record[5]))) \
    .filter(lambda order_cat_qty: order_cat_qty[1] == 'Furniture' and order_cat_qty[2] > 1) \
    .map(lambda order_cat_qty: (order_cat_qty[0], order_cat_qty[2]))

print(furniture_bulk_orders_rdd.collect())

[('5', 4), ('6', 2), ('7', 2), ('9', 2), ('14', 3), ('17', 3), ('20', 4), ('24', 4), ('32', 2), ('39', 3), ('44', 4), ('46', 3), ('52', 3), ('53', 4), ('59', 4), ('62', 5), ('64', 3), ('68', 3), ('76', 3), ('84', 4), ('85', 4), ('86', 4), ('91', 3), ('92', 2), ('94', 3), ('98', 2), ('99', 2)]


---
## Q10 — Order Count per Category
Using `map()` and `reduceByKey()`, count the total number of orders for each category.

In [ ]:
# Key by category (index 3), value = 1 per order, then sum
order_count_per_category_rdd = parsed_sales_rdd \
    .map(lambda record: (record[3], 1)) \
    .reduceByKey(lambda count_a, count_b: count_a + count_b)

print(order_count_per_category_rdd.collect())

[('Electronics', 66), ('Furniture', 34)]


---
## Q11 — Total Revenue per Customer
Using `map()` and `reduceByKey()`, calculate the total revenue generated by each customer.


In [ ]:
# Key by customer_id (index 1), value = quantity * unit_price
total_revenue_per_customer_rdd = parsed_sales_rdd \
    .map(lambda record: (record[1], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_customer_rdd.collect())

[('C112', 160000), ('C137', 10000), ('C150', 389000), ('C113', 133000), ('C103', 177000), ('C144', 39000), ('C107', 384000), ('C125', 224000), ('C124', 310000), ('C128', 27000), ('C118', 25000), ('C102', 161000), ('C119', 263000), ('C108', 48000), ('C114', 36000), ('C105', 54000), ('C120', 86000), ('C106', 25000), ('C109', 30000), ('C122', 60000), ('C141', 96000), ('C134', 480000), ('C147', 325000), ('C143', 79000), ('C123', 170000), ('C111', 60000), ('C129', 106000), ('C148', 48000), ('C110', 204000), ('C116', 85000), ('C140', 308000), ('C135', 244000), ('C133', 22000), ('C145', 153000), ('C101', 168000), ('C138', 7000), ('C132', 60000), ('C127', 140000), ('C149', 72000), ('C100', 16000), ('C136', 25000), ('C115', 60000), ('C130', 36000)]


---
## Q12 — Distinct Products Sold in Delhi
Using `filter()` and `map()`, find all unique products sold in Delhi.

In [ ]:
# Filter records for Delhi (index 4), extract product name (index 2), deduplicate
distinct_products_in_delhi_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], record[4])) \
    .filter(lambda product_city: product_city[1] == "Delhi") \
    .map(lambda product_city: product_city[0]) \
    .distinct()

print(distinct_products_in_delhi_rdd.collect())

['Headphones', 'Chair', 'Table', 'Tablet', 'Printer', 'Phone', 'Monitor', 'Laptop', 'Bed']


---
## Q13 — Total Quantity Sold per City
Using `map()` and `reduceByKey()`, find the total quantity sold in each city.

In [ ]:
# Key by city (index 4), value = quantity (index 5)
total_quantity_per_city_rdd = parsed_sales_rdd \
    .map(lambda record: (record[4], int(record[5]))) \
    .reduceByKey(lambda qty_a, qty_b: qty_a + qty_b)

print(total_quantity_per_city_rdd.collect())

[('Chennai', 33), ('Delhi', 44), ('Patna', 19), ('Hyderabad', 34), ('Mumbai', 47), ('Bangalore', 49), ('Pune', 45), ('Kolkata', 22)]


---
## Q14 — Orders with Total Sales Value Greater Than 50,000
Using `filter()`, extract all orders where the total sales value (`quantity × unit_price`) exceeds 50,000.

In [ ]:
# Map to (order_id, total_sales), filter where total_sales > 50000
high_value_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], int(record[5]) * int(record[6]))) \
    .filter(lambda order_sales: order_sales[1] > 50000)

print(high_value_orders_rdd.collect())

[('1', 60000), ('4', 125000), ('5', 72000), ('8', 110000), ('10', 150000), ('13', 120000), ('17', 54000), ('19', 100000), ('20', 72000), ('25', 125000), ('26', 100000), ('29', 125000), ('30', 90000), ('31', 60000), ('35', 60000), ('39', 54000), ('44', 80000), ('45', 100000), ('47', 275000), ('48', 100000), ('52', 60000), ('55', 220000), ('58', 165000), ('59', 72000), ('62', 90000), ('63', 120000), ('64', 60000), ('65', 60000), ('66', 125000), ('69', 150000), ('76', 60000), ('77', 120000), ('78', 150000), ('79', 60000), ('80', 110000), ('82', 120000), ('84', 80000), ('85', 72000), ('87', 60000), ('88', 100000), ('94', 54000), ('95', 60000), ('96', 125000), ('97', 120000)]


---
## Q15 — Total Revenue per Category
Using `map()` and `reduceByKey()`, calculate the total revenue generated for each category.

In [ ]:
# Key by category (index 3), value = quantity * unit_price
total_revenue_per_category_rdd = parsed_sales_rdd \
    .map(lambda record: (record[3], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_category_rdd.collect())

[('Electronics', 4346000), ('Furniture', 1259000)]


---
## Q16 — Total Sales per Customer
Using `map()` and `reduceByKey()`, calculate the total sales amount for each customer.

Output: `(customer_id, total_sales)`

In [ ]:
# Key by customer_id (index 1), value = quantity * unit_price
total_sales_per_customer_rdd = parsed_sales_rdd \
    .map(lambda record: (record[1], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda sales_a, sales_b: sales_a + sales_b)

print(total_sales_per_customer_rdd.collect())

[('C112', 160000), ('C137', 10000), ('C150', 389000), ('C113', 133000), ('C103', 177000), ('C144', 39000), ('C107', 384000), ('C125', 224000), ('C124', 310000), ('C128', 27000), ('C118', 25000), ('C102', 161000), ('C119', 263000), ('C108', 48000), ('C114', 36000), ('C105', 54000), ('C120', 86000), ('C106', 25000), ('C109', 30000), ('C122', 60000), ('C141', 96000), ('C134', 480000), ('C147', 325000), ('C143', 79000), ('C123', 170000), ('C111', 60000), ('C129', 106000), ('C148', 48000), ('C110', 204000), ('C116', 85000), ('C140', 308000), ('C135', 244000), ('C133', 22000), ('C145', 153000), ('C101', 168000), ('C138', 7000), ('C132', 60000), ('C127', 140000), ('C149', 72000), ('C100', 16000), ('C136', 25000), ('C115', 60000), ('C130', 36000)]


---
## Q17 — Orders from Delhi
Using `filter()`, extract all orders where city is **Delhi**.

Output: complete record

In [ ]:
# Filter records where city (index 4) is Delhi
orders_from_delhi_rdd = parsed_sales_rdd.filter(lambda record: record[4] == "Delhi")

print(orders_from_delhi_rdd.collect())

[['2', 'C137', 'Headphones', 'Electronics', 'Delhi', '5', '2000'], ['9', 'C125', 'Chair', 'Furniture', 'Delhi', '2', '4000'], ['14', 'C119', 'Table', 'Furniture', 'Delhi', '3', '7000'], ['21', 'C106', 'Tablet', 'Electronics', 'Delhi', '1', '25000'], ['22', 'C109', 'Headphones', 'Electronics', 'Delhi', '4', '2000'], ['23', 'C122', 'Printer', 'Electronics', 'Delhi', '4', '12000'], ['30', 'C147', 'Phone', 'Electronics', 'Delhi', '3', '30000'], ['31', 'C141', 'Monitor', 'Electronics', 'Delhi', '4', '15000'], ['33', 'C123', 'Monitor', 'Electronics', 'Delhi', '3', '15000'], ['46', 'C103', 'Table', 'Furniture', 'Delhi', '3', '7000'], ['58', 'C134', 'Laptop', 'Electronics', 'Delhi', '3', '55000'], ['67', 'C102', 'Tablet', 'Electronics', 'Delhi', '1', '25000'], ['89', 'C136', 'Tablet', 'Electronics', 'Delhi', '1', '25000'], ['96', 'C123', 'Tablet', 'Electronics', 'Delhi', '5', '25000'], ['99', 'C130', 'Bed', 'Furniture', 'Delhi', '2', '18000']]


---
## Q18 — Count Orders per City
Using `map()` and `reduceByKey()`, calculate the number of orders placed in each city.

Output: `(city, order_count)`

In [ ]:
# Key by city (index 4), value = 1 per order, then sum
order_count_per_city_rdd = parsed_sales_rdd \
    .map(lambda record: (record[4], 1)) \
    .reduceByKey(lambda count_a, count_b: count_a + count_b)

print(order_count_per_city_rdd.collect())

[('Chennai', 11), ('Delhi', 15), ('Patna', 11), ('Hyderabad', 10), ('Mumbai', 18), ('Bangalore', 15), ('Pune', 13), ('Kolkata', 7)]


---
## Q19 — Total Revenue per Product
Using `map()` and `reduceByKey()`, calculate total revenue (`quantity × unit_price`) per product.

Output: `(product, total_revenue)`

In [ ]:
# Key by product (index 2), value = quantity * unit_price
total_revenue_per_product_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_product_rdd.collect())

[('Printer', 408000), ('Headphones', 58000), ('Phone', 1260000), ('Tablet', 1350000), ('Bed', 666000), ('Table', 161000), ('Laptop', 880000), ('Chair', 52000), ('Monitor', 390000), ('Sofa', 380000)]


---
## Q20 — High Quantity Orders (qty > 3)
Using `filter()`, extract all records where quantity is greater than 3.

Output: complete record

In [ ]:
# Filter records where quantity (index 5) > 3
high_quantity_orders_rdd = parsed_sales_rdd.filter(lambda record: int(record[5]) > 3)

print(high_quantity_orders_rdd.collect())

[['1', 'C112', 'Printer', 'Electronics', 'Chennai', '5', '12000'], ['2', 'C137', 'Headphones', 'Electronics', 'Delhi', '5', '2000'], ['4', 'C113', 'Tablet', 'Electronics', 'Hyderabad', '5', '25000'], ['5', 'C150', 'Bed', 'Furniture', 'Hyderabad', '4', '18000'], ['10', 'C124', 'Phone', 'Electronics', 'Bangalore', '5', '30000'], ['13', 'C102', 'Phone', 'Electronics', 'Pune', '4', '30000'], ['18', 'C113', 'Headphones', 'Electronics', 'Bangalore', '4', '2000'], ['19', 'C124', 'Tablet', 'Electronics', 'Pune', '4', '25000'], ['20', 'C120', 'Bed', 'Furniture', 'Bangalore', '4', '18000'], ['22', 'C109', 'Headphones', 'Electronics', 'Delhi', '4', '2000'], ['23', 'C122', 'Printer', 'Electronics', 'Delhi', '4', '12000'], ['24', 'C102', 'Chair', 'Furniture', 'Bangalore', '4', '4000'], ['25', 'C150', 'Tablet', 'Electronics', 'Chennai', '5', '25000'], ['26', 'C119', 'Tablet', 'Electronics', 'Pune', '4', '25000'], ['29', 'C134', 'Tablet', 'Electronics', 'Mumbai', '5', '25000'], ['31', 'C141', 'Monito

---
## Q21 — Total Quantity Sold per Category
Using `map()` and `reduceByKey()`, calculate total quantity sold for each category.

Output: `(category, total_quantity)`

In [ ]:
# Key by category (index 3), value = quantity (index 5)
total_quantity_per_category_rdd = parsed_sales_rdd \
    .map(lambda record: (record[3], int(record[5]))) \
    .reduceByKey(lambda qty_a, qty_b: qty_a + qty_b)

print(total_quantity_per_category_rdd.collect())

[('Electronics', 201), ('Furniture', 92)]


---
## Q22 — Expensive Product Orders (> ₹25,000)
Using `filter()`, extract all orders where unit price exceeds 25,000.

Output: complete record

In [ ]:
# Filter records where unit_price (index 6) > 25000
expensive_product_orders_rdd = parsed_sales_rdd.filter(lambda record: int(record[6]) > 25000)

print(expensive_product_orders_rdd.collect())

[['3', 'C150', 'Phone', 'Electronics', 'Patna', '1', '30000'], ['8', 'C107', 'Laptop', 'Electronics', 'Bangalore', '2', '55000'], ['10', 'C124', 'Phone', 'Electronics', 'Bangalore', '5', '30000'], ['13', 'C102', 'Phone', 'Electronics', 'Pune', '4', '30000'], ['30', 'C147', 'Phone', 'Electronics', 'Delhi', '3', '30000'], ['37', 'C124', 'Phone', 'Electronics', 'Patna', '1', '30000'], ['47', 'C140', 'Laptop', 'Electronics', 'Pune', '5', '55000'], ['55', 'C147', 'Laptop', 'Electronics', 'Bangalore', '4', '55000'], ['58', 'C134', 'Laptop', 'Electronics', 'Delhi', '3', '55000'], ['63', 'C107', 'Phone', 'Electronics', 'Chennai', '4', '30000'], ['69', 'C150', 'Phone', 'Electronics', 'Mumbai', '5', '30000'], ['77', 'C101', 'Phone', 'Electronics', 'Chennai', '4', '30000'], ['78', 'C110', 'Phone', 'Electronics', 'Pune', '5', '30000'], ['80', 'C134', 'Laptop', 'Electronics', 'Mumbai', '2', '55000'], ['82', 'C103', 'Phone', 'Electronics', 'Pune', '4', '30000'], ['87', 'C127', 'Phone', 'Electronics'

---
 ## Q23 — Average Price per Product
Using `map()` and `reduceByKey()`, compute total price and count per product, then derive average.

Output: `(product, average_price)`

In [ ]:
# Key by product (index 2), value = (unit_price, 1); then sum to get (total_price, count)
average_price_per_product_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], (int(record[6]), 1))) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .mapValues(lambda totals: round(totals[0] / totals[1], 2))

print(average_price_per_product_rdd.collect())

[('Printer', 12000.0), ('Headphones', 2000.0), ('Phone', 30000.0), ('Tablet', 25000.0), ('Bed', 18000.0), ('Table', 7000.0), ('Laptop', 55000.0), ('Chair', 4000.0), ('Monitor', 15000.0), ('Sofa', 20000.0)]


---
## Q24 — Total Orders per Product
Using `map()` and `reduceByKey()`, calculate how many times each product was ordered.

Output: `(product, total_orders)`

In [ ]:
# Key by product (index 2), value = 1 per order, then sum
total_orders_per_product_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], 1)) \
    .reduceByKey(lambda count_a, count_b: count_a + count_b)

print(total_orders_per_product_rdd.collect())

[('Printer', 12), ('Headphones', 9), ('Phone', 12), ('Tablet', 17), ('Bed', 12), ('Table', 11), ('Laptop', 5), ('Chair', 4), ('Monitor', 11), ('Sofa', 7)]


---
## Q25 — Total Revenue per City
Using `map()` and `reduceByKey()`, calculate total revenue (`quantity × unit_price`) from each city.

Output: `(city, total_revenue)`

In [ ]:
# Key by city (index 4), value = quantity * unit_price
total_revenue_per_city_rdd = parsed_sales_rdd \
    .map(lambda record: (record[4], int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_city_rdd.collect())

[('Chennai', 687000), ('Delhi', 712000), ('Patna', 294000), ('Hyderabad', 597000), ('Mumbai', 903000), ('Bangalore', 1001000), ('Pune', 1157000), ('Kolkata', 254000)]


---
## Q26 — Total Orders per Customer
Using `map()` and `reduceByKey()`, calculate the number of orders placed by each customer.

Output: `(customer_id, total_orders)`

In [ ]:
# Key by customer_id (index 1), value = 1 per order, then sum
total_orders_per_customer_rdd = parsed_sales_rdd \
    .map(lambda record: (record[1], 1)) \
    .reduceByKey(lambda count_a, count_b: count_a + count_b)

print(total_orders_per_customer_rdd.collect())

[('C112', 2), ('C137', 1), ('C150', 6), ('C113', 2), ('C103', 3), ('C144', 2), ('C107', 6), ('C125', 4), ('C124', 4), ('C128', 3), ('C118', 1), ('C102', 3), ('C119', 6), ('C108', 2), ('C114', 1), ('C105', 1), ('C120', 2), ('C106', 1), ('C109', 3), ('C122', 2), ('C141', 2), ('C134', 4), ('C147', 3), ('C143', 3), ('C123', 2), ('C111', 1), ('C129', 2), ('C148', 2), ('C110', 2), ('C116', 3), ('C140', 3), ('C135', 3), ('C133', 2), ('C145', 2), ('C101', 2), ('C138', 1), ('C132', 1), ('C127', 2), ('C149', 1), ('C100', 1), ('C136', 1), ('C115', 1), ('C130', 1)]


---
## Q27 — Low Price Orders (< ₹5,000)
Using `filter()`, extract all orders where unit price is less than 5,000.

Output: `order_id`

In [ ]:
# Map to (order_id, unit_price), filter where unit_price < 5000, extract order_id
low_price_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], int(record[6]))) \
    .filter(lambda order_price: order_price[1] < 5000) \
    .map(lambda order_price: order_price[0])

print(low_price_orders_rdd.collect())

['2', '9', '11', '18', '22', '24', '34', '36', '40', '60', '71', '86', '91']


---
## Q28 — Total Revenue per (Category, City)
Using `map()` and `reduceByKey()`, calculate total revenue for each `(category, city)` pair.

Output: `((category, city), total_revenue)`

In [ ]:
# Key by (category, city) — indices (3, 4); value = quantity * unit_price
total_revenue_per_category_city_rdd = parsed_sales_rdd \
    .map(lambda record: ((record[3], record[4]), int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_category_city_rdd.collect())

[(('Electronics', 'Chennai'), 501000), (('Electronics', 'Delhi'), 626000), (('Electronics', 'Patna'), 198000), (('Electronics', 'Hyderabad'), 345000), (('Furniture', 'Hyderabad'), 252000), (('Furniture', 'Mumbai'), 306000), (('Electronics', 'Bangalore'), 814000), (('Furniture', 'Delhi'), 86000), (('Electronics', 'Pune'), 1091000), (('Electronics', 'Kolkata'), 174000), (('Electronics', 'Mumbai'), 597000), (('Furniture', 'Chennai'), 186000), (('Furniture', 'Bangalore'), 187000), (('Furniture', 'Pune'), 66000), (('Furniture', 'Patna'), 96000), (('Furniture', 'Kolkata'), 80000)]


---
## Q29 — Products Sold in Mumbai
Using `filter()` and `map()`, list all product names sold in Mumbai.

Output: `product_name`

In [ ]:
# Filter records for Mumbai (index 4), extract product name (index 2)
products_sold_in_mumbai_rdd = parsed_sales_rdd \
    .map(lambda record: (record[2], record[4])) \
    .filter(lambda product_city: product_city[1] == "Mumbai") \
    .map(lambda product_city: product_city[0])

print(products_sold_in_mumbai_rdd.collect())

['Bed', 'Table', 'Printer', 'Tablet', 'Tablet', 'Sofa', 'Headphones', 'Bed', 'Sofa', 'Monitor', 'Phone', 'Headphones', 'Printer', 'Table', 'Laptop', 'Sofa', 'Chair', 'Table']


---
## Q30 — Total Quantity per Customer
Using `map()` and `reduceByKey()`, calculate total quantity purchased by each customer.

Output: `(customer_id, total_quantity)`

In [ ]:
# Key by customer_id (index 1), value = quantity (index 5)
total_quantity_per_customer_rdd = parsed_sales_rdd \
    .map(lambda record: (record[1], int(record[5]))) \
    .reduceByKey(lambda qty_a, qty_b: qty_a + qty_b)

print(total_quantity_per_customer_rdd.collect())

[('C112', 9), ('C137', 5), ('C150', 21), ('C113', 9), ('C103', 9), ('C144', 3), ('C107', 14), ('C125', 13), ('C124', 12), ('C128', 3), ('C118', 1), ('C102', 9), ('C119', 15), ('C108', 4), ('C114', 3), ('C105', 3), ('C120', 6), ('C106', 1), ('C109', 6), ('C122', 7), ('C141', 7), ('C134', 14), ('C147', 8), ('C143', 5), ('C123', 8), ('C111', 4), ('C129', 7), ('C148', 3), ('C110', 8), ('C116', 9), ('C140', 9), ('C135', 12), ('C133', 2), ('C145', 9), ('C101', 8), ('C138', 1), ('C132', 4), ('C127', 6), ('C149', 4), ('C100', 4), ('C136', 1), ('C115', 5), ('C130', 2)]


---
## Q31 — Orders with Even Quantity
Using `filter()`, extract all records where quantity sold is an even number.

Output: `(customer_id, quantity)`

In [ ]:
# Map to (customer_id, quantity), filter where quantity is even
even_quantity_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[1], int(record[5]))) \
    .filter(lambda customer_qty: customer_qty[1] % 2 == 0)

print(even_quantity_orders_rdd.collect())

[('C150', 4), ('C103', 2), ('C144', 2), ('C107', 2), ('C125', 2), ('C102', 4), ('C108', 2), ('C113', 4), ('C124', 4), ('C120', 4), ('C109', 4), ('C122', 4), ('C102', 4), ('C119', 4), ('C141', 4), ('C143', 2), ('C111', 4), ('C134', 4), ('C135', 4), ('C129', 4), ('C124', 2), ('C145', 4), ('C143', 2), ('C147', 4), ('C101', 4), ('C107', 4), ('C107', 4), ('C125', 4), ('C107', 2), ('C101', 4), ('C132', 4), ('C134', 2), ('C103', 4), ('C127', 4), ('C149', 4), ('C100', 4), ('C127', 2), ('C112', 4), ('C108', 2), ('C148', 2), ('C125', 4), ('C120', 2), ('C130', 2)]


---
## Q32 — Total Sales per (Product, City)
Using `map()` and `reduceByKey()`, calculate total sales for each `(product, city)` pair.

Output: `((product, city), total_sales)`

In [ ]:
# Key by (product, city) — indices (2, 4); value = quantity * unit_price
total_sales_per_product_city_rdd = parsed_sales_rdd \
    .map(lambda record: ((record[2], record[4]), int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda sales_a, sales_b: sales_a + sales_b)

print(total_sales_per_product_city_rdd.collect())

[(('Printer', 'Chennai'), 96000), (('Headphones', 'Delhi'), 18000), (('Phone', 'Patna'), 120000), (('Tablet', 'Hyderabad'), 225000), (('Bed', 'Hyderabad'), 144000), (('Bed', 'Mumbai'), 126000), (('Table', 'Mumbai'), 28000), (('Laptop', 'Bangalore'), 330000), (('Chair', 'Delhi'), 8000), (('Phone', 'Bangalore'), 270000), (('Headphones', 'Patna'), 2000), (('Tablet', 'Chennai'), 150000), (('Phone', 'Pune'), 390000), (('Table', 'Delhi'), 42000), (('Printer', 'Kolkata'), 24000), (('Printer', 'Mumbai'), 48000), (('Bed', 'Chennai'), 126000), (('Headphones', 'Bangalore'), 14000), (('Tablet', 'Pune'), 375000), (('Bed', 'Bangalore'), 144000), (('Tablet', 'Delhi'), 200000), (('Printer', 'Delhi'), 48000), (('Chair', 'Bangalore'), 16000), (('Printer', 'Patna'), 36000), (('Monitor', 'Chennai'), 15000), (('Tablet', 'Mumbai'), 225000), (('Phone', 'Delhi'), 90000), (('Monitor', 'Delhi'), 105000), (('Table', 'Pune'), 14000), (('Headphones', 'Kolkata'), 20000), (('Monitor', 'Hyderabad'), 60000), (('Printe

---
## Q33 — Order Count per (Category, City)
Using `map()` and `reduceByKey()`, count orders for each `(category, city)` pair.

Output: `((category, city), order_count)`

In [ ]:
# Key by (category, city) — indices (3, 4); value = 1 per order, then sum
order_count_per_category_city_rdd = parsed_sales_rdd \
    .map(lambda record: ((record[3], record[4]), 1)) \
    .reduceByKey(lambda count_a, count_b: count_a + count_b)

print(order_count_per_category_city_rdd.collect())

[(('Electronics', 'Chennai'), 8), (('Electronics', 'Delhi'), 11), (('Electronics', 'Patna'), 7), (('Electronics', 'Hyderabad'), 5), (('Furniture', 'Hyderabad'), 5), (('Furniture', 'Mumbai'), 9), (('Electronics', 'Bangalore'), 10), (('Furniture', 'Delhi'), 4), (('Electronics', 'Pune'), 10), (('Electronics', 'Kolkata'), 6), (('Electronics', 'Mumbai'), 9), (('Furniture', 'Chennai'), 3), (('Furniture', 'Bangalore'), 5), (('Furniture', 'Pune'), 3), (('Furniture', 'Patna'), 4), (('Furniture', 'Kolkata'), 1)]


---
## Q34 — Orders Where Total Value < ₹20,000
Using `filter()`, extract all orders where `quantity × unit_price` is less than 20,000.

Output: `order_id`

In [ ]:
# Map to (order_id, total_value), filter where total_value < 20000, extract order_id
low_value_orders_rdd = parsed_sales_rdd \
    .map(lambda record: (record[0], int(record[5]) * int(record[6]))) \
    .filter(lambda order_value: order_value[1] < 20000) \
    .map(lambda order_value: order_value[0])

print(low_value_orders_rdd.collect())

['2', '7', '9', '11', '18', '22', '24', '28', '32', '34', '36', '38', '40', '41', '42', '43', '49', '57', '60', '71', '73', '74', '75', '81', '86', '91', '93', '98', '100']


---
## Q35 — Total Revenue per (Customer, City)
Using `map()` and `reduceByKey()`, calculate total revenue for each `(customer_id, city)` pair.

Output: `((customer_id, city), total_revenue)`

In [ ]:
# Key by (customer_id, city) — indices (1, 4); value = quantity * unit_price
total_revenue_per_customer_city_rdd = parsed_sales_rdd \
    .map(lambda record: ((record[1], record[4]), int(record[5]) * int(record[6]))) \
    .reduceByKey(lambda revenue_a, revenue_b: revenue_a + revenue_b)

print(total_revenue_per_customer_city_rdd.collect())

[(('C112', 'Chennai'), 60000), (('C137', 'Delhi'), 10000), (('C150', 'Patna'), 30000), (('C113', 'Hyderabad'), 125000), (('C150', 'Hyderabad'), 72000), (('C103', 'Mumbai'), 36000), (('C144', 'Mumbai'), 14000), (('C107', 'Bangalore'), 232000), (('C125', 'Delhi'), 8000), (('C124', 'Bangalore'), 180000), (('C128', 'Patna'), 9000), (('C118', 'Chennai'), 25000), (('C102', 'Pune'), 120000), (('C119', 'Delhi'), 21000), (('C108', 'Kolkata'), 24000), (('C114', 'Mumbai'), 36000), (('C105', 'Chennai'), 54000), (('C113', 'Bangalore'), 8000), (('C124', 'Pune'), 100000), (('C120', 'Bangalore'), 72000), (('C106', 'Delhi'), 25000), (('C109', 'Delhi'), 8000), (('C122', 'Delhi'), 48000), (('C102', 'Bangalore'), 16000), (('C150', 'Chennai'), 125000), (('C119', 'Pune'), 100000), (('C141', 'Patna'), 36000), (('C119', 'Chennai'), 75000), (('C134', 'Mumbai'), 235000), (('C147', 'Delhi'), 90000), (('C141', 'Delhi'), 60000), (('C143', 'Pune'), 79000), (('C123', 'Delhi'), 170000), (('C150', 'Kolkata'), 10000), 